In [ ]:
# CELL 1 — Install Dependencies

import subprocess, sys

packages = [
    "sentence-transformers",
    "faiss-cpu",
    "fastapi",
    "uvicorn[standard]",
    "nest-asyncio",
    "pydantic",
    "numpy",
    "httpx",
    "tabulate",
]

print("📦 Menginstall dependencies...\n")
for pkg in packages:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", pkg, "-q"],
        capture_output=True, text=True
    )
    status = "✅" if result.returncode == 0 else "❌"
    print(f"  {status} {pkg}")

print("\n🎉 Semua dependency berhasil diinstall!")
print("⚠️  Jika ini pertama kali, RESTART KERNEL sebelum lanjut ke cell berikutnya.")

📦 Menginstall dependencies...

  ✅ sentence-transformers
  ✅ faiss-cpu
  ✅ fastapi
  ✅ uvicorn[standard]
  ✅ nest-asyncio
  ✅ pydantic
  ✅ numpy
  ✅ httpx
  ✅ tabulate

🎉 Semua dependency berhasil diinstall!
⚠️  Jika ini pertama kali, RESTART KERNEL sebelum lanjut ke cell berikutnya.


In [1]:
# CELL 2 — Imports & Konfigurasi Global

import json
import os
import warnings
import numpy as np
from pathlib import Path
from typing import List, Optional, Dict, Any
from dataclasses import dataclass, field
from collections import Counter

warnings.filterwarnings("ignore")

# ── Konfigurasi Path ─────────────────────────────────────────
BASE_DIR     = Path(os.getcwd())
DATASET_PATH = BASE_DIR / "data" / "products.json"

# ── Konfigurasi Model ────────────────────────────────────────
MODEL_NAME    = "BAAI/bge-m3"
TOP_N_DEFAULT = 5

# ── Konfigurasi Hybrid Scoring ───────────────────────────────
COMPLEMENTARY_WEIGHT = 0.40   # Bobot skor komplementer kategori
SEMANTIC_WEIGHT      = 0.60   # Bobot cosine similarity embedding

print("✅ Imports berhasil.")
print(f"   Base Dir     : {BASE_DIR}")
print(f"   Dataset Path : {DATASET_PATH}")
print(f"   Model        : {MODEL_NAME}")
print(f"   Hybrid Score : {COMPLEMENTARY_WEIGHT*100:.0f}% Complementary + {SEMANTIC_WEIGHT*100:.0f}% Semantic")

✅ Imports berhasil.
   Base Dir     : d:\Project\Hackathon
   Dataset Path : d:\Project\Hackathon\data\products.json
   Model        : BAAI/bge-m3
   Hybrid Score : 40% Complementary + 60% Semantic


In [2]:
# CELL 3 — Load Dataset

@dataclass
class Product:
    """
    Representasi satu produk UMKM.
    Field-field disesuaikan dengan format data fullstack Tenantify.
    """
    id          : str
    tenantId    : str
    tenantName  : str
    boothNumber : str
    name        : str
    category    : str
    price       : int
    description : str
    rating      : float
    image       : str
    soldCount   : int
    isPopular   : bool

    @property
    def text_for_embedding(self) -> str:
        """
        Menggabungkan seluruh informasi produk menjadi satu string
        kaya konteks untuk diproses model embedding.
        """
        return (
            f"{self.name}. "
            f"{self.description} "
            f"Kategori: {self.category}. "
            f"Dari toko: {self.tenantName}."
        )


def load_dataset(path: Path) -> List[Product]:
    """Membaca products.json dan mengkonversi ke list of Product."""
    if not path.exists():
        raise FileNotFoundError(
            f"Dataset tidak ditemukan di: {path}\n"
            "Pastikan file data/products.json ada di direktori proyek."
        )
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)
    return [Product(**item) for item in raw]


# Load dataset
products = load_dataset(DATASET_PATH)

# Ringkasan dataset
category_counts = Counter(p.category for p in products)
tenant_counts   = Counter(p.tenantId for p in products)

print(f"📦 Dataset berhasil dimuat!")
print(f"   Total Produk : {len(products)}")
print(f"   Total Tenant : {len(tenant_counts)}")
print(f"\n📊 Distribusi Kategori:")
for cat, count in sorted(category_counts.items(), key=lambda x: -x[1]):
    bar = "█" * count
    print(f"   {cat:<15} {bar} ({count})")

print(f"\n🏪 Contoh Produk (format fullstack):")
for p in products[:3]:
    print(f"   [{p.id}] {p.name}")
    print(f"          → {p.tenantName} | {p.boothNumber} | Rp{p.price:,} | ⭐{p.rating} | 🔥{p.isPopular}")

📦 Dataset berhasil dimuat!
   Total Produk : 48
   Total Tenant : 19

📊 Distribusi Kategori:
   Minuman         █████████████ (13)
   Camilan         █████████████ (13)
   Makanan         ███████████ (11)
   Gorengan        ██████ (6)
   Dessert         █████ (5)

🏪 Contoh Produk (format fullstack):
   [cp-1] Kopi Susu Gula Aren
          → Kopi & Roti Nusantara | Blok A-04 | Rp18,000 | ⭐4.9 | 🔥True
   [cp-2] Roti Bakar Nutella Keju
          → Kopi & Roti Nusantara | Blok A-04 | Rp15,000 | ⭐4.7 | 🔥True
   [cp-3] Es Kopi Americano
          → Kopi & Roti Nusantara | Blok A-04 | Rp14,000 | ⭐4.6 | 🔥False


In [3]:
# ============================================================
# CELL 4 — Embedder: Wrapper BAAI/bge-m3
# ⚠️ Download model ~2.3GB pada pertama kali dijalankan.
# ============================================================

import faiss
from sentence_transformers import SentenceTransformer


class Embedder:
    """
    Wrapper untuk model BAAI/bge-m3 (multilingual, support Bahasa Indonesia).
    Tugasnya:
      1. Load model dari HuggingFace (sekali saja)
      2. Encode teks menjadi dense vector
      3. Precompute & simpan embedding semua produk
      4. Build FAISS index untuk pencarian cepat
    """

    def __init__(self, model_name: str = MODEL_NAME):
        print(f"⏳ Memuat model {model_name}...")
        print("   (Download ~2.3GB jika pertama kali — harap sabar)\n")
        self.model = SentenceTransformer(model_name)
        self.embeddings: Optional[np.ndarray] = None
        self.index: Optional[faiss.IndexFlatIP] = None
        print(f"✅ Model berhasil dimuat!")
        print(f"   Embedding Dim : {self.model.get_sentence_embedding_dimension()}")

    def encode(self, texts: List[str], batch_size: int = 32, show_progress: bool = True) -> np.ndarray:
        """Encode list of teks → numpy array of L2-normalized embeddings."""
        vecs = self.model.encode(
            texts,
            batch_size=batch_size,
            normalize_embeddings=True,   # L2-normalize → cosine sim = inner product
            show_progress_bar=show_progress,
        )
        return vecs.astype(np.float32)

    def build_index(self, product_list: List[Product]) -> None:
        """
        Precompute embedding semua produk dan build FAISS index.
        Dipanggil sekali saat inisialisasi sistem.
        """
        print(f"\n🔄 Membangun index untuk {len(product_list)} produk...")
        texts = [p.text_for_embedding for p in product_list]
        self.embeddings = self.encode(texts, show_progress=True)

        dim = self.embeddings.shape[1]
        self.index = faiss.IndexFlatIP(dim)   # Inner Product index
        self.index.add(self.embeddings)

        print(f"✅ FAISS index selesai dibangun!")
        print(f"   Vektor tersimpan : {self.index.ntotal}")
        print(f"   Dimensi          : {dim}")

    def get_similarities(self, query_idx: int) -> np.ndarray:
        """
        Hitung cosine similarity antara produk di query_idx
        dengan semua produk lain di index.
        """
        if self.index is None:
            raise RuntimeError("Index belum dibangun. Panggil build_index() terlebih dahulu.")
        query_vec = self.embeddings[query_idx:query_idx + 1]
        scores, _ = self.index.search(query_vec, self.index.ntotal)
        return scores[0]

    def encode_query(self, text: str) -> np.ndarray:
        """Encode satu teks query untuk pencarian ad-hoc."""
        return self.encode([text], show_progress=False)[0]

    def get_similarities_from_vector(self, query_vec: np.ndarray) -> np.ndarray:
        """Hitung similarity dari vector yang sudah di-encode."""
        scores, _ = self.index.search(query_vec.reshape(1, -1), self.index.ntotal)
        return scores[0]


# Inisialisasi Embedder & build index
embedder = Embedder(model_name=MODEL_NAME)
embedder.build_index(products)


⏳ Memuat model BAAI/bge-m3...
   (Download ~2.3GB jika pertama kali — harap sabar)



modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

✅ Model berhasil dimuat!
   Embedding Dim : 1024

🔄 Membangun index untuk 48 produk...


Batches:   0%|          | 0/2 [00:01<?, ?it/s]

✅ FAISS index selesai dibangun!
   Vektor tersimpan : 48
   Dimensi          : 1024


In [4]:
# ============================================================
# CELL 5 — Recommender: Core Hybrid Scoring Engine
# ============================================================

# ── Complementary Category Map ───────────────────────────────
# Kategori mengikuti format fullstack: Minuman, Makanan, Camilan, Gorengan, Dessert
# Score 1.0 = sangat komplementer | 0.0 = tidak relevan
COMPLEMENTARY_MAP: Dict[tuple, float] = {
    # Camilan ↔ Minuman (paling cocok)
    ("Camilan",  "Minuman")  : 1.0,
    ("Camilan",  "Dessert")  : 0.6,
    ("Camilan",  "Gorengan") : 0.3,

    # Gorengan ↔ Minuman
    ("Gorengan", "Minuman")  : 1.0,
    ("Gorengan", "Dessert")  : 0.5,
    ("Gorengan", "Camilan")  : 0.3,

    # Makanan ↔ Minuman & Dessert
    ("Makanan",  "Minuman")  : 0.9,
    ("Makanan",  "Dessert")  : 0.8,
    ("Makanan",  "Camilan")  : 0.4,
    ("Makanan",  "Gorengan") : 0.3,

    # Dessert ↔ Minuman
    ("Dessert",  "Minuman")  : 1.0,
    ("Dessert",  "Camilan")  : 0.5,

    # Minuman ↔ Minuman (kompetitif, bukan komplementer)
    ("Minuman",  "Minuman")  : 0.15,
}


def get_complementary_score(cat_a: str, cat_b: str) -> float:
    """Ambil skor komplementer antara dua kategori (dua arah)."""
    if cat_a == cat_b:
        # Kategori sama: sedikit relevan tapi bukan prioritas
        return COMPLEMENTARY_MAP.get((cat_a, cat_b), 0.2)
    score = COMPLEMENTARY_MAP.get((cat_a, cat_b))
    if score is None:
        score = COMPLEMENTARY_MAP.get((cat_b, cat_a), 0.1)
    return score


@dataclass
class RecommendationResult:
    """Satu item hasil rekomendasi beserta skor dan penjelasannya."""
    product             : Product
    hybrid_score        : float
    semantic_score      : float
    complementary_score : float
    reason              : str


class Recommender:
    """
    Core Recommendation Engine untuk Tenantify.

    hybrid_score = (COMPLEMENTARY_WEIGHT × complementary_score)
                 + (SEMANTIC_WEIGHT      × semantic_similarity)

    Constraint utama:
      - Produk dari tenantId BERBEDA (pemerataan pedagang)
    """

    def __init__(self, product_list: List[Product], embedder: Embedder):
        self.products     = product_list
        self.embedder     = embedder
        # Mapping id → index untuk lookup O(1)
        self.id_to_idx    = {p.id: i for i, p in enumerate(product_list)}
        # Mapping name → product untuk lookup dari order items
        self.name_to_product: Dict[str, Product] = {p.name: p for p in product_list}
        print("✅ Recommender siap digunakan.")

    def recommend(
        self,
        product_id: str,
        top_n: int = TOP_N_DEFAULT,
        exclude_same_tenant: bool = True,
        category_filter: Optional[str] = None,
        exclude_product_ids: Optional[List[str]] = None,
    ) -> List[RecommendationResult]:
        """
        Rekomendasi produk komplementer berdasarkan satu produk.

        Args:
            product_id          : ID produk yang dipilih user (format: 'cp-1', 'ag-1', dst)
            top_n               : Jumlah rekomendasi yang dikembalikan
            exclude_same_tenant : Jika True, tidak rekomendasikan dari tenant yang sama
            category_filter     : Jika diisi, hanya rekomendasikan dari kategori ini
            exclude_product_ids : Daftar product_id yang dikecualikan dari hasil
        """
        if product_id not in self.id_to_idx:
            raise ValueError(f"Produk '{product_id}' tidak ditemukan di dataset.")

        query_idx     = self.id_to_idx[product_id]
        query_product = self.products[query_idx]
        exclude_ids   = set(exclude_product_ids or [])
        exclude_ids.add(product_id)  # Selalu exclude diri sendiri

        sim_scores = self.embedder.get_similarities(query_idx)

        results: List[RecommendationResult] = []
        for idx, candidate in enumerate(self.products):
            if candidate.id in exclude_ids:
                continue
            if exclude_same_tenant and candidate.tenantId == query_product.tenantId:
                continue
            if category_filter and candidate.category != category_filter:
                continue

            comp_score = get_complementary_score(query_product.category, candidate.category)
            sem_score  = float(np.clip(sim_scores[idx], 0.0, 1.0))
            hybrid     = COMPLEMENTARY_WEIGHT * comp_score + SEMANTIC_WEIGHT * sem_score
            reason     = _build_reason(query_product, candidate, comp_score, sem_score)

            results.append(RecommendationResult(
                product             = candidate,
                hybrid_score        = round(hybrid, 4),
                semantic_score      = round(sem_score, 4),
                complementary_score = round(comp_score, 4),
                reason              = reason,
            ))

        results.sort(key=lambda r: r.hybrid_score, reverse=True)
        return results[:top_n]

    def recommend_from_order(
        self,
        order_items: List[Dict[str, Any]],
        top_n: int = TOP_N_DEFAULT,
    ) -> List[RecommendationResult]:
        """
        Rekomendasi berbasis pesanan aktif (order items dari fullstack).
        Mengagregasi embedding semua item yang sudah dipesan,
        lalu mencari produk komplementer yang belum ada di pesanan.

        Args:
            order_items : List item dari field 'items' pada order object fullstack
                          Format: [{ productName, tenantName, price, quantity, notes }, ...]
            top_n       : Jumlah rekomendasi yang dikembalikan
        """
        # Resolve order items ke Product objects
        ordered_products: List[Product] = []
        for item in order_items:
            product = self.name_to_product.get(item["productName"])
            if product:
                ordered_products.append(product)

        if not ordered_products:
            raise ValueError("Tidak ada produk dari order yang cocok dengan dataset.")

        # Kumpulkan semua tenantId & productId yang sudah ada di order
        ordered_tenant_ids  = {p.tenantId  for p in ordered_products}
        ordered_product_ids = {p.id        for p in ordered_products}

        # Agregasi embedding: rata-rata embedding semua produk yang dipesan
        ordered_idxs   = [self.id_to_idx[p.id] for p in ordered_products]
        ordered_vecs   = self.embedder.embeddings[ordered_idxs]
        mean_vec       = ordered_vecs.mean(axis=0)
        # L2-normalize ulang setelah rata-rata
        norm           = np.linalg.norm(mean_vec)
        mean_vec       = (mean_vec / norm).astype(np.float32)

        # Hitung similarity dari mean vector ke semua produk
        sim_scores = self.embedder.get_similarities_from_vector(mean_vec)

        # Kategori dominan dari order untuk complementary scoring
        category_counts = Counter(p.category for p in ordered_products)
        dominant_category = category_counts.most_common(1)[0][0]

        results: List[RecommendationResult] = []
        for idx, candidate in enumerate(self.products):
            # Skip produk yang sudah ada di order
            if candidate.id in ordered_product_ids:
                continue
            # Skip tenant yang sudah ada di order (pemerataan)
            if candidate.tenantId in ordered_tenant_ids:
                continue

            comp_score = get_complementary_score(dominant_category, candidate.category)
            sem_score  = float(np.clip(sim_scores[idx], 0.0, 1.0))
            hybrid     = COMPLEMENTARY_WEIGHT * comp_score + SEMANTIC_WEIGHT * sem_score

            reason = (
                f"{candidate.category} ini cocok melengkapi pesananmu "
                f"yang didominasi {dominant_category} "
                f"(relevansi: {int(sem_score*100)}%)."
            )

            results.append(RecommendationResult(
                product             = candidate,
                hybrid_score        = round(hybrid, 4),
                semantic_score      = round(sem_score, 4),
                complementary_score = round(comp_score, 4),
                reason              = reason,
            ))

        results.sort(key=lambda r: r.hybrid_score, reverse=True)
        return results[:top_n]


def _build_reason(query: Product, candidate: Product, comp: float, sem: float) -> str:
    """Buat kalimat penjelasan singkat mengapa produk ini direkomendasikan."""
    comp_label = (
        "sangat cocok dipadukan" if comp >= 0.9 else
        "cocok dipadukan"        if comp >= 0.7 else
        "cukup cocok dipadukan"  if comp >= 0.5 else
        "bisa dipadukan"
    )
    return (
        f"{candidate.category} ini {comp_label} dengan "
        f"{query.category} '{query.name}' "
        f"(relevansi konteks: {int(sem*100)}%)."
    )


# Inisialisasi Recommender
recommender = Recommender(product_list=products, embedder=embedder)

✅ Recommender siap digunakan.


In [5]:
# ============================================================
# CELL 6 — Pydantic Schemas (Request & Response)
# Format disesuaikan dengan data fullstack Tenantify
# ============================================================

from pydantic import BaseModel, Field


# ── Product Schema (sama persis dengan format fullstack) ─────
class ProductSchema(BaseModel):
    id          : str
    tenantId    : str
    tenantName  : str
    boothNumber : str
    name        : str
    category    : str
    price       : int
    description : str
    rating      : float
    image       : str
    soldCount   : int
    isPopular   : bool


# ── Order Item Schema (dari field 'items' di order object) ───
class OrderItemSchema(BaseModel):
    productName : str  = Field(..., description="Nama produk")
    tenantName  : str  = Field(..., description="Nama tenant")
    price       : int  = Field(..., description="Harga satuan")
    quantity    : int  = Field(1,   description="Jumlah")
    notes       : str  = Field("",  description="Catatan dari pembeli")


# ── Recommendation Item ──────────────────────────────────────
class RecommendationItem(BaseModel):
    product             : ProductSchema
    hybrid_score        : float = Field(description="Skor gabungan [0–1]")
    semantic_score      : float = Field(description="Cosine similarity embedding [0–1]")
    complementary_score : float = Field(description="Skor kesesuaian kategori [0–1]")
    reason              : str   = Field(description="Alasan rekomendasi")


# ── Request Bodies ───────────────────────────────────────────
class RecommendByProductRequest(BaseModel):
    """Request rekomendasi berdasarkan satu produk."""
    product_id           : str           = Field(...,  description="ID produk yang dipilih", example="gk-1")
    top_n                : int           = Field(5,    ge=1, le=20)
    exclude_same_tenant  : bool          = Field(True, description="Exclude produk dari tenant yang sama")
    category_filter      : Optional[str] = Field(None, description="Filter kategori: Minuman/Makanan/Camilan/Gorengan/Dessert")


class RecommendByOrderRequest(BaseModel):
    """Request rekomendasi berdasarkan pesanan aktif (order items)."""
    items : List[OrderItemSchema] = Field(..., description="List item pesanan dari order object")
    top_n : int                   = Field(5,   ge=1, le=20)


# ── Response Bodies ──────────────────────────────────────────
class RecommendByProductResponse(BaseModel):
    query_product   : ProductSchema
    recommendations : List[RecommendationItem]
    total_returned  : int


class RecommendByOrderResponse(BaseModel):
    ordered_products : List[ProductSchema]
    recommendations  : List[RecommendationItem]
    total_returned   : int


class HealthResponse(BaseModel):
    status        : str
    model         : str
    total_products: int
    total_tenants : int


def product_to_schema(p: Product) -> ProductSchema:
    return ProductSchema(
        id=p.id, tenantId=p.tenantId, tenantName=p.tenantName,
        boothNumber=p.boothNumber, name=p.name, category=p.category,
        price=p.price, description=p.description, rating=p.rating,
        image=p.image, soldCount=p.soldCount, isPopular=p.isPopular
    )


print("✅ Pydantic schemas berhasil didefinisikan.")
print("   Schemas yang tersedia:")
print("     • ProductSchema             (sama persis dengan format fullstack)")
print("     • OrderItemSchema           (dari field items di order object)")
print("     • RecommendByProductRequest / Response")
print("     • RecommendByOrderRequest   / Response")
print("     • HealthResponse")

✅ Pydantic schemas berhasil didefinisikan.
   Schemas yang tersedia:
     • ProductSchema             (sama persis dengan format fullstack)
     • OrderItemSchema           (dari field items di order object)
     • RecommendByProductRequest / Response
     • RecommendByOrderRequest   / Response
     • HealthResponse


In [6]:
# ============================================================
# CELL 7 — FastAPI App + Jalankan Server di Notebook
# ============================================================

import nest_asyncio
import uvicorn
import threading
import time
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware

# Patch event loop agar FastAPI bisa jalan di dalam Jupyter
nest_asyncio.apply()

# ── Inisialisasi FastAPI ─────────────────────────────────────
app = FastAPI(
    title       = "Tenantify AI — Recommendation API",
    description = (
        "Sistem rekomendasi produk UMKM berbasis Content-Based Filtering "
        "menggunakan model `BAAI/bge-m3`. Dirancang untuk pemerataan peluang tenant UMKM."
    ),
    version  = "1.0.0",
    docs_url = "/docs",
    redoc_url= "/redoc",
)

# ── CORS (untuk integrasi frontend fullstack) ────────────────
app.add_middleware(
    CORSMiddleware,
    allow_origins  = ["*"],
    allow_methods  = ["*"],
    allow_headers  = ["*"],
)


# ── Endpoints ────────────────────────────────────────────────

@app.get("/health", response_model=HealthResponse, tags=["System"])
def health_check():
    """Cek status server, model, dan statistik dataset."""
    tenant_ids = {p.tenantId for p in products}
    return HealthResponse(
        status        = "ok",
        model         = MODEL_NAME,
        total_products= len(products),
        total_tenants = len(tenant_ids),
    )


@app.get("/products", response_model=List[ProductSchema], tags=["Products"])
def list_products(
    category  : Optional[str] = None,
    tenant_id : Optional[str] = None,
    is_popular: Optional[bool] = None,
):
    """
    Ambil semua produk. Bisa difilter berdasarkan:
    - **category**: Minuman | Makanan | Camilan | Gorengan | Dessert
    - **tenant_id**: ID tenant (misal: tenant-1)
    - **is_popular**: true / false
    """
    filtered = products
    if category:
        filtered = [p for p in filtered if p.category == category]
    if tenant_id:
        filtered = [p for p in filtered if p.tenantId == tenant_id]
    if is_popular is not None:
        filtered = [p for p in filtered if p.isPopular == is_popular]
    return [product_to_schema(p) for p in filtered]


@app.get("/products/{product_id}", response_model=ProductSchema, tags=["Products"])
def get_product(product_id: str):
    """Ambil detail satu produk berdasarkan ID (contoh: `gk-1`, `cp-1`, `ag-1`)."""
    idx = recommender.id_to_idx.get(product_id)
    if idx is None:
        raise HTTPException(status_code=404, detail=f"Produk '{product_id}' tidak ditemukan.")
    return product_to_schema(products[idx])


@app.post("/recommend/by-product", response_model=RecommendByProductResponse, tags=["Recommendation"])
def recommend_by_product(req: RecommendByProductRequest):
    """
    **Rekomendasi berdasarkan produk yang dipilih.**

    Gunakan ketika user sedang melihat detail produk dan ingin
    menampilkan seksi *'Produk lain yang cocok dinikmati bersama'*.

    **Contoh request:**
    ```json
    { "product_id": "gk-1", "top_n": 5, "exclude_same_tenant": true }
    ```
    """
    try:
        results = recommender.recommend(
            product_id          = req.product_id,
            top_n               = req.top_n,
            exclude_same_tenant = req.exclude_same_tenant,
            category_filter     = req.category_filter,
        )
    except ValueError as e:
        raise HTTPException(status_code=404, detail=str(e))

    query_idx = recommender.id_to_idx[req.product_id]
    return RecommendByProductResponse(
        query_product   = product_to_schema(products[query_idx]),
        recommendations = [
            RecommendationItem(
                product             = product_to_schema(r.product),
                hybrid_score        = r.hybrid_score,
                semantic_score      = r.semantic_score,
                complementary_score = r.complementary_score,
                reason              = r.reason,
            ) for r in results
        ],
        total_returned = len(results),
    )


@app.post("/recommend/by-order", response_model=RecommendByOrderResponse, tags=["Recommendation"])
def recommend_by_order(req: RecommendByOrderRequest):
    """
    **Rekomendasi berdasarkan pesanan aktif.**

    Gunakan ketika user sudah ada pesanan aktif dan ingin
    menampilkan seksi *'Tambahkan ke pesananmu'*.
    Menerima langsung field `items` dari order object fullstack.

    **Contoh request (langsung pakai items dari order object):**
    ```json
    {
      "items": [
        { "productName": "Kopi Susu Gula Aren", "tenantName": "Kopi & Roti Nusantara",
          "price": 18000, "quantity": 1, "notes": "" },
        { "productName": "Paket Ayam Geprek Krispi + Nasi", "tenantName": "Ayam Geprek Sambal Korek",
          "price": 25000, "quantity": 1, "notes": "Level pedas sedang" }
      ],
      "top_n": 5
    }
    ```
    """
    try:
        results = recommender.recommend_from_order(
            order_items = [item.model_dump() for item in req.items],
            top_n       = req.top_n,
        )
    except ValueError as e:
        raise HTTPException(status_code=422, detail=str(e))

    # Resolve ordered products untuk ditampilkan di response
    ordered_products = [
        p for item in req.items
        if (p := recommender.name_to_product.get(item.productName))
    ]

    return RecommendByOrderResponse(
        ordered_products = [product_to_schema(p) for p in ordered_products],
        recommendations  = [
            RecommendationItem(
                product             = product_to_schema(r.product),
                hybrid_score        = r.hybrid_score,
                semantic_score      = r.semantic_score,
                complementary_score = r.complementary_score,
                reason              = r.reason,
            ) for r in results
        ],
        total_returned = len(results),
    )


# ── Jalankan Server di background thread ─────────────────────
PORT = 8000

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(2)  # Tunggu server siap

print("🚀 Tenantify AI Server berjalan!")
print(f"   Base URL     : http://localhost:{PORT}")
print(f"   Swagger UI   : http://localhost:{PORT}/docs   ← Coba semua endpoint di sini!")
print(f"   ReDoc        : http://localhost:{PORT}/redoc")
print(f"\n   Endpoints:")
print(f"     GET  /health                  → Status server")
print(f"     GET  /products                → List semua produk")
print(f"     GET  /products/{{id}}           → Detail produk")
print(f"     POST /recommend/by-product    → Rekomendasi dari 1 produk")
print(f"     POST /recommend/by-order      → Rekomendasi dari pesanan aktif")

🚀 Tenantify AI Server berjalan!
   Base URL     : http://localhost:8000
   Swagger UI   : http://localhost:8000/docs   ← Coba semua endpoint di sini!
   ReDoc        : http://localhost:8000/redoc

   Endpoints:
     GET  /health                  → Status server
     GET  /products                → List semua produk
     GET  /products/{id}           → Detail produk
     POST /recommend/by-product    → Rekomendasi dari 1 produk
     POST /recommend/by-order      → Rekomendasi dari pesanan aktif


In [7]:
# ============================================================
# CELL 8 — Demo: Rekomendasi Berdasarkan Produk
# ============================================================

import httpx
from tabulate import tabulate

BASE_URL = f"http://localhost:{PORT}"


def demo_by_product(product_id: str, top_n: int = 5):
    """Demo endpoint POST /recommend/by-product"""
    resp = httpx.post(f"{BASE_URL}/recommend/by-product", json={
        "product_id"         : product_id,
        "top_n"              : top_n,
        "exclude_same_tenant": True,
    })
    data = resp.json()
    q    = data["query_product"]

    print("=" * 72)
    print(f"🛍️  PRODUK YANG DIPILIH USER")
    print(f"   [{q['id']}] {q['name']}")
    print(f"   Tenant      : {q['tenantName']} ({q['boothNumber']})")
    print(f"   Kategori    : {q['category']} | Harga: Rp{q['price']:,} | ⭐ {q['rating']}")
    print(f"   Deskripsi   : {q['description'][:75]}...")
    print("=" * 72)
    print(f"\n🎯 TOP {top_n} REKOMENDASI PRODUK KOMPLEMENTER:\n")

    rows = []
    for i, item in enumerate(data["recommendations"], 1):
        p = item["product"]
        rows.append([
            i, p["id"], p["name"][:28], p["category"],
            f"Rp{p['price']:,}", f"⭐{p['rating']}",
            p["boothNumber"],
            f"{item['hybrid_score']:.3f}",
            f"{item['complementary_score']:.2f}",
            f"{item['semantic_score']:.3f}",
        ])

    print(tabulate(
        rows,
        headers=["#", "ID", "Nama Produk", "Kategori", "Harga",
                 "Rating", "Booth", "Hybrid ▼", "Kompl.", "Semantik"],
        tablefmt="rounded_outline"
    ))

    print(f"\n💡 ALASAN REKOMENDASI:")
    for i, item in enumerate(data["recommendations"], 1):
        p = item["product"]
        print(f"   {i}. [{p['id']}] {p['name']}: {item['reason']}")
    print()


# Demo 1: Kue Ikan Isi Coklat (gk-1)
print("━" * 72)
print("DEMO 1 — /recommend/by-product : Kue Ikan Isi Coklat")
print("━" * 72)
demo_by_product("gk-1", top_n=5)

# Demo 2: Paket Ayam Geprek (ag-1)
print("━" * 72)
print("DEMO 2 — /recommend/by-product : Paket Ayam Geprek Krispi + Nasi")
print("━" * 72)
demo_by_product("ag-1", top_n=5)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
DEMO 1 — /recommend/by-product : Kue Ikan Isi Coklat
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🛍️  PRODUK YANG DIPILIH USER
   [gk-1] Kue Ikan Isi Coklat
   Tenant      : Gorengan & Kue Ikan Kak Sari (Blok B-07)
   Kategori    : Camilan | Harga: Rp5,000 | ⭐ 4.8
   Deskripsi   : Kue ikan crispy berbentuk ikan dengan isian coklat lumer, dibuat dari adona...

🎯 TOP 5 REKOMENDASI PRODUK KOMPLEMENTER:

╭─────┬──────┬─────────────────────┬────────────┬──────────┬──────────┬───────────┬────────────┬──────────┬────────────╮
│   # │ ID   │ Nama Produk         │ Kategori   │ Harga    │ Rating   │ Booth     │   Hybrid ▼ │   Kompl. │   Semantik │
├─────┼──────┼─────────────────────┼────────────┼──────────┼──────────┼───────────┼────────────┼──────────┼────────────┤
│   1 │ cp-1 │ Kopi Susu Gula Aren │ Minuman    │ Rp18,000 │ ⭐4.9    │ Blok A-04 │      1     │        1 │      1     │
│   2 │ cp-3

In [8]:
# ============================================================
# CELL 9 — Demo: Rekomendasi Berdasarkan Pesanan Aktif
# Simulasi order object dari fullstack Tenantify
# ============================================================

def demo_by_order(order: dict, top_n: int = 5):
    """Demo endpoint POST /recommend/by-order menggunakan order object fullstack."""
    resp = httpx.post(f"{BASE_URL}/recommend/by-order", json={
        "items": order["items"],
        "top_n": top_n,
    })
    data = resp.json()

    print("=" * 72)
    print(f"🧾 PESANAN AKTIF  [Order ID: {order['id']} | Kode: {order['pickupCode']}]")
    print(f"   {order['orderType']} — {order.get('tableNumber', '-')}")
    for it in order["items"]:
        print(f"   • {it['productName']} x{it['quantity']}  →  {it['tenantName']}  (Rp{it['price']:,})")
    print(f"   Total: Rp{order['total']:,} | Status: {order['status']}")
    print("=" * 72)
    print(f"\n🎯 TOP {top_n} REKOMENDASI 'TAMBAHKAN KE PESANANMU':\n")

    rows = []
    for i, item in enumerate(data["recommendations"], 1):
        p = item["product"]
        rows.append([
            i, p["id"], p["name"][:28], p["category"],
            f"Rp{p['price']:,}", f"⭐{p['rating']}",
            p["boothNumber"],
            f"{item['hybrid_score']:.3f}",
        ])

    print(tabulate(
        rows,
        headers=["#", "ID", "Nama Produk", "Kategori", "Harga", "Rating", "Booth", "Score ▼"],
        tablefmt="rounded_outline"
    ))

    print(f"\n💡 ALASAN:")
    for i, item in enumerate(data["recommendations"], 1):
        p = item["product"]
        print(f"   {i}. [{p['id']}] {p['name']}: {item['reason']}")
    print()


# ── Simulasi Order Object dari fullstack ─────────────────────
sample_order = {
    "id"         : "ORD-9201",
    "pickupCode" : "TNT-501",
    "orderType"  : "Dine-in (Makan di Tempat)",
    "tableNumber": "Meja 12",
    "tenantNames": ["Kopi & Roti Nusantara", "Ayam Geprek Sambal Korek"],
    "items": [
        {
            "productName": "Kopi Susu Gula Aren",
            "tenantName" : "Kopi & Roti Nusantara",
            "price"      : 18000,
            "quantity"   : 1,
            "notes"      : "Gula aren sedikit",
        },
        {
            "productName": "Paket Ayam Geprek Krispi + Nasi",
            "tenantName" : "Ayam Geprek Sambal Korek",
            "price"      : 25000,
            "quantity"   : 1,
            "notes"      : "Level pedas sedang (level 2)",
        },
    ],
    "subtotal"     : 43000,
    "platformFee"  : 1000,
    "total"        : 44000,
    "status"       : "Siap Diambil",
    "paymentMethod": "QRIS",
    "createdAt"    : "8 menit lalu",
}

# Demo 3: Rekomendasi dari order aktif
print("━" * 72)
print("DEMO 3 — /recommend/by-order : Simulasi Order Object dari Fullstack")
print("━" * 72)
demo_by_order(sample_order, top_n=5)

# Demo 4: Order lain — hanya camilan
camilan_order = {
    "id"         : "ORD-9202",
    "pickupCode" : "TNT-502",
    "orderType"  : "Take Away",
    "tableNumber": "-",
    "tenantNames": ["Gorengan & Kue Ikan Kak Sari"],
    "items": [
        {
            "productName": "Kue Ikan Isi Coklat",
            "tenantName" : "Gorengan & Kue Ikan Kak Sari",
            "price"      : 5000,
            "quantity"   : 3,
            "notes"      : "",
        },
        {
            "productName": "Cireng Isi Keju Mozarella",
            "tenantName" : "Gorengan & Kue Ikan Kak Sari",
            "price"      : 10000,
            "quantity"   : 1,
            "notes"      : "Extra keju",
        },
    ],
    "subtotal"     : 25000,
    "platformFee"  : 1000,
    "total"        : 26000,
    "status"       : "Sedang Diproses",
    "paymentMethod": "QRIS",
    "createdAt"    : "2 menit lalu",
}

print("━" * 72)
print("DEMO 4 — /recommend/by-order : Pesanan Hanya Camilan")
print("━" * 72)
demo_by_order(camilan_order, top_n=5)

print("━" * 72)
print("✅ Semua demo selesai!")
print(f"   Swagger UI: http://localhost:{PORT}/docs")
print("━" * 72)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
DEMO 3 — /recommend/by-order : Simulasi Order Object dari Fullstack
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🧾 PESANAN AKTIF  [Order ID: ORD-9201 | Kode: TNT-501]
   Dine-in (Makan di Tempat) — Meja 12
   • Kopi Susu Gula Aren x1  →  Kopi & Roti Nusantara  (Rp18,000)
   • Paket Ayam Geprek Krispi + Nasi x1  →  Ayam Geprek Sambal Korek  (Rp25,000)
   Total: Rp44,000 | Status: Siap Diambil

🎯 TOP 5 REKOMENDASI 'TAMBAHKAN KE PESANANMU':

╭─────┬──────┬───────────────────────────┬────────────┬─────────┬──────────┬───────────┬───────────╮
│   # │ ID   │ Nama Produk               │ Kategori   │ Harga   │ Rating   │ Booth     │   Score ▼ │
├─────┼──────┼───────────────────────────┼────────────┼─────────┼──────────┼───────────┼───────────┤
│   1 │ jp-1 │ Klepon Pandan Original    │ Camilan    │ Rp8,000 │ ⭐4.9    │ Blok A-09 │     0.814 │
│   2 │ jp-2 │ Onde-Onde Wijen           │ Camilan   